<div style="background-color: #f8f9fa; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);">
  <div style="display: flex; justify-content: space-between; align-items: center;">
    <img src="https://sigmoidal.ai/wp-content/uploads/2024/09/Academia-Sigmoidal-Light.png" alt="Academia Sigmoidal Logo" width="250" height="auto">
    <div style="text-align: right;">
<h1 style="color: #007bff; margin: 0; font-size: 24px;">Pos-Graduacao em Visao Computacional</h1>
    </div>
</div>
<hr style="border: none; height: 1px; background-color: #007bff; margin: 20px 0;">
<h3 style="color: #343a40; margin: 0; font-size: 20px;"><strong>VIS101: Fundamentos da Visao Computacional</strong></h3>
<p style="color: #6c757d; margin: 5px 0 0; font-size: 14px;"><strong>Instrutor:</strong> Carlos Melo, MSc.</p>
</div>

# Chroma key: colocando um objeto em qualquer cenário

O chroma key é a técnica de gravar diante de um fundo de cor uniforme (o clássico fundo verde) para depois **remover essa cor** e colocar qualquer cenário no lugar. É a aplicação direta da segmentação por cor: em vez de isolar um objeto, isola-se o **fundo** pela cor e o que sobra é a pessoa.

Neste notebook o trabalho é feito sobre um vídeo. Além da segmentação, entram dois pontos novos: **como carregar um vídeo no OpenCV** e **como salvar o resultado em um novo vídeo**.

## Preparando o ambiente

A célula abaixo baixa os dois vídeos usados na aula: a gravação em fundo verde e o cenário de fundo.

In [ ]:
!mkdir -p data
!wget -q https://raw.githubusercontent.com/carlosfab/visao-computacional/main/vis101/chroma-key-video/data/greenscreen.mp4 -O data/greenscreen.mp4
!wget -q https://raw.githubusercontent.com/carlosfab/visao-computacional/main/vis101/chroma-key-video/data/fundo-deserto.mp4 -O data/fundo-deserto.mp4

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})
print('Setup pronto.')

## Carregando um vídeo no OpenCV

Um vídeo nada mais é do que uma **sequência de imagens** (os quadros, ou *frames*), exibidas rápido o suficiente para dar a sensação de movimento. No OpenCV, o objeto `cv2.VideoCapture` abre o arquivo e permite ler um quadro de cada vez.

Cada quadro que sai do vídeo é exatamente uma imagem como as que já vínhamos usando: uma matriz em **BGR**.

In [ ]:
# Abrir o arquivo de vídeo
cap = cv2.VideoCapture('data/greenscreen.mp4')

# Ler o primeiro quadro. read() devolve dois valores:
#   ok    -> True se conseguiu ler um quadro, False se o vídeo acabou
#   frame -> a imagem (matriz BGR), ou None se não houve leitura
ok, frame = cap.read()
print('leu?', ok, '| formato do quadro:', frame.shape)

In [ ]:
# O quadro é uma imagem BGR; converter para RGB só para o matplotlib exibir certo
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

Alguns metadados úteis do vídeo: a taxa de quadros por segundo (**FPS**) e o total de quadros. O FPS será necessário na hora de salvar o resultado com a mesma velocidade.

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print('FPS:', round(fps, 1), '| total de quadros:', total)

Para processar o vídeo inteiro, os quadros são lidos em um laço até `read()` devolver `False`. Aqui todos os quadros são guardados em uma lista. Ao final, `release()` fecha o arquivo.

In [ ]:
cap = cv2.VideoCapture('data/greenscreen.mp4')
frames = []
while True:
    ok, frame = cap.read()
    if not ok:           # acabou o vídeo
        break
    frames.append(frame)
cap.release()
print('quadros carregados:', len(frames))

## Removendo o verde

A ideia do chroma key é criar uma máscara do **fundo verde** e ficar apenas com o que **não** é verde. Como já visto, o espaço HSV torna a seleção por cor mais estável, então a máscara do verde sai de um `cv2.inRange`.

In [ ]:
frame = frames[len(frames) // 2]          # um quadro do meio do vídeo
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

# Faixa do verde do fundo (H em torno de 35 a 90)
verde = cv2.inRange(hsv, (35, 70, 50), (90, 255, 255))

plt.imshow(verde, cmap='gray')
plt.axis('off')
plt.show()

A máscara acima marca o fundo. Invertendo-a, marca-se o que interessa: a pessoa. Para descartar pequenos ruídos e restos de cenário, mantém-se apenas o **maior objeto** da máscara.

In [ ]:
pessoa = cv2.bitwise_not(verde)           # o que NAO e verde

# manter so o maior contorno (a pessoa)
cnts, _ = cv2.findContours(pessoa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
mask = np.zeros_like(pessoa)
if cnts:
    maior = max(cnts, key=cv2.contourArea)
    cv2.drawContours(mask, [maior], -1, 255, -1)

plt.imshow(mask, cmap='gray')
plt.axis('off')
plt.show()

Uma leve suavização na borda da máscara reduz o serrilhado antes da composição.

In [ ]:
mask = cv2.GaussianBlur(mask, (7, 7), 0)

## Compondo sobre o cenário

Com a máscara da pessoa em mãos, a composição é uma média ponderada pixel a pixel: onde a máscara vale 1, aparece a pessoa; onde vale 0, aparece o fundo. A máscara suavizada (valores entre 0 e 1) faz a transição na borda.

In [ ]:
cap = cv2.VideoCapture('data/fundo-deserto.mp4')
ok, fundo = cap.read()
cap.release()
fundo = cv2.resize(fundo, (frame.shape[1], frame.shape[0]))

alpha = (mask / 255.0)[..., None]
composto = (frame * alpha + fundo * (1 - alpha)).astype(np.uint8)

plt.imshow(cv2.cvtColor(composto, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## Aplicando ao vídeo inteiro

A função abaixo reúne os passos: recebe um quadro do fundo verde e o quadro correspondente do cenário, e devolve a composição.

In [ ]:
def compor(frame, fundo):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    verde = cv2.inRange(hsv, (35, 70, 50), (90, 255, 255))
    pessoa = cv2.bitwise_not(verde)
    cnts, _ = cv2.findContours(pessoa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(pessoa)
    if cnts:
        cv2.drawContours(mask, [max(cnts, key=cv2.contourArea)], -1, 255, -1)
    h, w = mask.shape
    mask = cv2.GaussianBlur(mask, (7, 7), 0)
    alpha = (mask / 255.0)[..., None]
    fundo = cv2.resize(fundo, (frame.shape[1], frame.shape[0]))
    return (frame * alpha + fundo * (1 - alpha)).astype(np.uint8)

Carregando também os quadros do cenário, para combinar cada quadro do fundo verde com o quadro correspondente do deserto.

In [ ]:
cap = cv2.VideoCapture('data/fundo-deserto.mp4')
fundo_frames = []
while True:
    ok, f = cap.read()
    if not ok:
        break
    fundo_frames.append(f)
cap.release()

n = min(len(frames), len(fundo_frames))
print('quadros a compor:', n)

## Salvando o resultado em vídeo

Para gravar um novo vídeo, usa-se `cv2.VideoWriter`. Ele precisa de quatro informações: o nome do arquivo, o **codec** (via `VideoWriter_fourcc`), o **FPS** e o **tamanho do quadro** (largura, altura). Depois, cada quadro composto é escrito com `write`, e no fim `release` fecha o arquivo.

In [ ]:
h, w = frames[0].shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
saida = cv2.VideoWriter('resultado.mp4', fourcc, fps, (w, h))

for i in range(n):
    quadro = compor(frames[i], fundo_frames[i])
    saida.write(quadro)

saida.release()
print('video salvo em resultado.mp4')

No Colab, o arquivo gerado pode ser baixado direto para o computador.

In [ ]:
from google.colab import files
files.download('resultado.mp4')

Está fechado o ciclo: um vídeo é carregado quadro a quadro, cada quadro tem o fundo verde removido pela cor e substituído por um cenário, e o resultado é salvo como um novo vídeo. A mesma segmentação por cor, agora como efeito visual.